# LeetCode 855 — Exam Room bằng Python

Link đề: https://leetcode.com/problems/exam-room/ · Độ khó: **Medium** · Chủ đề: Design (thiết kế lớp), Ordered Set (tập có thứ tự), Heap (Priority Queue, hàng đợi ưu tiên)

Skeleton LeetCode:

```python
class ExamRoom:

    def __init__(self, n: int):
        ...

    def seat(self) -> int:
        ...

    def leave(self, p: int) -> None:
        ...


# Your ExamRoom object will be instantiated and called as such:
# obj = ExamRoom(n)
# param_1 = obj.seat()
# obj.leave(p)
```

Ghi chú: đây là bài **thiết kế cấu trúc dữ liệu** có tới `n = 10^9` ghế nên **không thể** lưu từng ghế; ta chỉ lưu các **học sinh đang ngồi** và các **khoảng trống** giữa họ. Khi ngồi, học sinh luôn chọn ghế **xa người gần nhất** nhất (và số nhỏ nhất nếu hòa); khi rời đi, hai khoảng trống kề nhau **gộp** lại. Notebook tự dựng bộ chạy chuỗi lệnh (như LeetCode) để kiểm tra từng cách, và dùng một oracle duyệt từng ghế trên phòng nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: phòng thi một hàng `n` ghế; `seat()` chọn ghế **tối đa khoảng cách tới người gần nhất** (hòa: ghế số nhỏ nhất; phòng trống: ghế 0); `leave(p)` cho người ở ghế `p` rời đi.
2. Nắm các công cụ Python cần thiết: lớp có trạng thái riêng (`__init__`), `heapq` (đống nhỏ nhất, **xóa lười**), `bisect.insort`/`bisect_left` để duy trì danh sách có thứ tự, từ điển láng giềng, ký tự gác (sentinel) `-1` và `n`, so sánh tuple để phá hòa.
3. Hiểu cách **biểu diễn bằng khoảng trống** `(l, r)` giữa hai người (hoặc hai "gác") và công thức ghế tối ưu: `0` nếu `l = -1`, `n - 1` nếu `r = n`, ngược lại `(l + r) // 2` với khoảng cách `(r - l) // 2`.
4. Đi qua 4 cách: duyệt từng ghế (định nghĩa), duyệt các khoảng trống giữa người đang ngồi, đống ưu tiên có xóa lười, tập có thứ tự mô phỏng bằng `bisect`; tự dựng **bộ chạy chuỗi lệnh** để kiểm chứng.
5. Tránh các lỗi kinh điển: làm tròn lên ở giữa (không phải ghế số nhỏ nhất), chia đôi cả hai đầu, quên phá hòa theo ghế nhỏ nhất, dùng phần tử cũ trong đống (xóa lười không kiểm tra), và dùng đống nhỏ nhất với khoảng cách dương.

## 2. Đề bài & ví dụ

Có một phòng thi `n` ghế xếp một hàng, đánh số từ `0` tới `n - 1`.

Khi một học sinh vào phòng, họ **phải** ngồi vào ghế **tối đa hóa khoảng cách tới người gần nhất**. Nếu có nhiều ghế như vậy, họ ngồi ghế có **số nhỏ nhất**. Nếu phòng chưa có ai thì họ ngồi ghế `0`.

Thiết kế lớp `ExamRoom`:

- `ExamRoom(int n)` khởi tạo phòng với `n` ghế.
- `int seat()` trả về số ghế mà học sinh tiếp theo sẽ ngồi.
- `void leave(int p)` cho học sinh đang ngồi ở ghế `p` rời đi. Đảm bảo có học sinh ở ghế `p`.

**Ví dụ 1**

```text
Input
["ExamRoom", "seat", "seat", "seat", "seat", "leave", "seat"]
[[10], [], [], [], [], [4], []]
Output
[null, 0, 9, 4, 2, null, 5]
```

- `seat()` -> 0 (phòng trống, ngồi ghế 0).
- `seat()` -> 9 (ghế cuối, xa người ở 0 nhất).
- `seat()` -> 4 (giữa 0 và 9, cách 4 hoặc 5; ghế 4 số nhỏ hơn).
- `seat()` -> 2 (giữa 0 và 4).
- `leave(4)`.
- `seat()` -> 5 (ghế 5 nằm giữa người ở 2 và người ở 9, cách người gần nhất 3 ghế; xem tính tay ở mục 4).

**Ràng buộc:**

- `1 <= n <= 10^9`
- Đảm bảo có học sinh ngồi ở ghế `p` khi gọi `leave(p)`.
- Tối đa `10^4` lần gọi `seat` và `leave`.

In [1]:
from typing import List, Optional
import heapq
from bisect import bisect_left, insort
import random, time

def run_ops(cls, ops: List[str], args: list) -> list:
    """Mô phỏng cách LeetCode gọi: phần tử đầu là tên lớp (constructor), các phần tử sau là tên method.
    Trả về list kết quả, None ứng với null."""
    obj = None
    out = []
    for i, (op, arg) in enumerate(zip(ops, args)):
        if i == 0:
            assert op == cls.__name__, (op, cls.__name__)
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

example_ops = ["ExamRoom", "seat", "seat", "seat", "seat", "leave", "seat"]
example_args = [[10], [], [], [], [], [4], []]
example_expected = [None, 0, 9, 4, 2, None, 5]
print("ops     =", example_ops)
print("kỳ vọng =", example_expected)

ops     = ['ExamRoom', 'seat', 'seat', 'seat', 'seat', 'leave', 'seat']
kỳ vọng = [None, 0, 9, 4, 2, None, 5]


## 3. Kiến thức nền

### 3.1. Lớp và trạng thái riêng

Trong bài thiết kế, LeetCode tạo `ExamRoom(n)` rồi gọi liên tiếp `seat`, `leave` lên **cùng một đối tượng**. Dữ liệu phải nằm trong `self` (khởi tạo ở `__init__`), **không** đặt ở cấp lớp (sẽ bị dùng chung giữa các đối tượng).

In [2]:
class Counter1:
    def __init__(self):
        self.n = 0            # riêng cho từng đối tượng
    def inc(self):
        self.n += 1
        return self.n

a, b = Counter1(), Counter1()
print(a.inc(), a.inc(), b.inc())
print(run_ops(Counter1, ["Counter1", "inc", "inc"], [[], [], []]))

1 2 1
[None, 1, 2]


### 3.2. Biểu diễn bằng khoảng trống và ký tự gác

Với `n` lên tới `10^9`, ta chỉ lưu các ghế **có người**. Giữa hai người liên tiếp ở `l < r` (các ghế `l + 1 .. r - 1` trống) là một **khoảng trống** `(l, r)`. Hai đầu hàng được "gác" bằng `-1` (bên trái ghế 0) và `n` (bên phải ghế `n - 1`): phòng trống là một khoảng trống duy nhất `(-1, n)`.

Với khoảng trống `(l, r)`, ghế tốt nhất và khoảng cách đạt được:

| loại | ghế | khoảng cách |
|---|---|---|
| `l == -1` (đầu hàng), `r != n` | `0` | `r` (= `r - l - 1`) |
| `r == n` (cuối hàng), `l != -1` | `n - 1` | `n - 1 - l` (= `r - l - 1`) |
| `l == -1` và `r == n` (phòng trống) | `0` | (mọi giá trị; chỉ có một khoảng trống) |
| giữa hai người | `(l + r) // 2` | `(r - l) // 2` |

In [3]:
def gap_info(l: int, r: int, n: int):
    """Trả về (khoảng cách, ghế) tốt nhất trong khoảng trống (l, r)."""
    if l == -1:
        return (r - l - 1, 0) if r != n else (n, 0)
    if r == n:
        return (r - l - 1, n - 1)
    return ((r - l) // 2, (l + r) // 2)

n = 10
for gap in [(-1, 10), (-1, 4), (0, 9), (0, 4), (4, 9), (9, 10), (2, 4), (4, 5)]:
    d, pos = gap_info(*gap, n)
    print(f"khoảng trống {gap}: khoảng cách {d}, ghế {pos}")

khoảng trống (-1, 10): khoảng cách 10, ghế 0
khoảng trống (-1, 4): khoảng cách 4, ghế 0
khoảng trống (0, 9): khoảng cách 4, ghế 4
khoảng trống (0, 4): khoảng cách 2, ghế 2
khoảng trống (4, 9): khoảng cách 2, ghế 6
khoảng trống (9, 10): khoảng cách 0, ghế 9
khoảng trống (2, 4): khoảng cách 1, ghế 3
khoảng trống (4, 5): khoảng cách 0, ghế 4


### 3.3. `heapq`: đống nhỏ nhất và xóa lười

`heapq` cài đặt đống **nhỏ nhất** (phần tử nhỏ nhất ở đầu): `heappush(h, x)`, `heappop(h)`. Để lấy **lớn nhất** ta lưu giá trị đảo dấu (`-d`). Hai tuple được so sánh từng phần tử: `(-d, l, r)` ưu tiên `d` lớn nhất, rồi `l` nhỏ nhất.

**Xóa lười (lazy deletion):** `heapq` không xóa phần tử bất kỳ. Thay vào đó ta để phần tử **cũ** (đã lỗi thời) trong đống và **kiểm tra khi lấy ra**: nếu không còn hợp lệ thì bỏ qua.

In [4]:
h = []
for item in [(-2, 4, 9), (-4, 0, 9), (-4, -1, 5), (-3, 0, 6)]:
    heapq.heappush(h, item)
print("lấy ra theo thứ tự:", [heapq.heappop(h) for _ in range(len(h))])
# (-4, -1, 5) trước (-4, 0, 9): cùng khoảng cách 4, l nhỏ hơn đứng trước -> ghế nhỏ hơn

lấy ra theo thứ tự: [(-4, -1, 5), (-4, 0, 9), (-3, 0, 6), (-2, 4, 9)]


### 3.4. `bisect`: danh sách có thứ tự

`bisect_left(a, x)` cho vị trí chèn `x` vào danh sách đã sắp xếp `a` để vẫn có thứ tự; `insort(a, x)` chèn luôn. Chèn vào list Python tốn `O(k)` (dịch phần tử) nhưng cực nhanh trong thực tế (`memmove`), còn tìm vị trí là `O(log k)`.

In [5]:
occ = [0, 4, 9]
insort(occ, 2)
print(occ, "| vị trí của 4:", bisect_left(occ, 4))
del occ[bisect_left(occ, 4)]
print(occ)

[0, 2, 4, 9] | vị trí của 4: 2
[0, 2, 9]


### 3.5. Dict láng giềng: gộp khoảng trống khi `leave`

Với mỗi người ở `p` ta nhớ **người liền trước** `prev[p]` và **liền sau** `next[p]` (hoặc ký tự gác). Khi `p` rời đi, hai khoảng trống `(prev[p], p)` và `(p, next[p])` **gộp** thành `(prev[p], next[p])`, và láng giềng của hai bên được nối lại.

In [6]:
prev, nxt = {0: -1, 4: 0, 9: 4}, {-1: 0, 0: 4, 4: 9, 9: 10}
p = 4
l, r = prev.pop(p), nxt.pop(p)
nxt[l], prev[r] = r, l
print("sau khi 4 rời đi: next =", nxt, "| prev =", prev)

sau khi 4 rời đi: next = {-1: 0, 0: 9, 9: 10} | prev = {0: -1, 9: 0}


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Chạy ví dụ 1 với `n = 10`, trạng thái = các khoảng trống:

| thao tác | khoảng trống trước | chọn | ghế | sau |
|---|---|---|---|---|
| `seat()` | `(-1, 10)` | phòng trống | 0 | `(0, 10)` |
| `seat()` | `(0, 10)` (cuối hàng: khoảng cách `9 - 0 = 9`) | cuối hàng | 9 | `(0, 9)`, `(9, 10)` |
| `seat()` | `(0, 9)` (giữa: `9 // 2 = 4`) | giữa | `(0 + 9) // 2 = 4` | `(0, 4)`, `(4, 9)`, `(9, 10)` |
| `seat()` | `(0, 4)`: 2; `(4, 9)`: 2; `(9, 10)`: 0 | hòa 2, `l` nhỏ nhất | `(0 + 4) // 2 = 2` | `(0, 2)`, `(2, 4)`, `(4, 9)`, `(9, 10)` |
| `leave(4)` | gộp `(2, 4)` và `(4, 9)` | | | `(0, 2)`, `(2, 9)`, `(9, 10)` |
| `seat()` | `(2, 9)`: `7 // 2 = 3`; `(0, 2)`: 1 | giữa | `(2 + 9) // 2 = 5` | ... |

Ghế 5 cách 3 người ở 2 và 4 người ở 9: khoảng cách tới người gần nhất là 3. Kết quả `[null, 0, 9, 4, 2, null, 5]` ✓.

Điểm mấu chốt: chỉ **các khoảng trống giữa hai người đang ngồi** quan trọng; mỗi lần ngồi **tách một khoảng trống thành hai**, mỗi lần rời đi **gộp hai thành một**.

In [7]:
def hand_run(n, steps):
    gaps = [(-1, n)]
    occ = []
    for kind, arg in steps:
        if kind == "seat":
            best = max(gaps, key=lambda g: (gap_info(*g, n)[0], -g[0]))     # khoảng cách lớn nhất, l nhỏ nhất
            d, pos = gap_info(*best, n)
            gaps.remove(best)
            gaps += [(best[0], pos), (pos, best[1])]
            gaps.sort()
            occ.append(pos)
            print(f"seat() -> {pos} (khoảng trống {best}, khoảng cách {d}); khoảng trống: {gaps}")
        else:
            idx = sorted(occ)
            i = idx.index(arg)
            l = idx[i - 1] if i > 0 else -1
            r = idx[i + 1] if i + 1 < len(idx) else n
            gaps.remove((l, arg)); gaps.remove((arg, r)); gaps.append((l, r)); gaps.sort()
            occ.remove(arg)
            print(f"leave({arg}): gộp ({l}, {arg}) và ({arg}, {r}); khoảng trống: {gaps}")

hand_run(10, [("seat", None)] * 4 + [("leave", 4), ("seat", None)])

seat() -> 0 (khoảng trống (-1, 10), khoảng cách 10); khoảng trống: [(-1, 0), (0, 10)]
seat() -> 9 (khoảng trống (0, 10), khoảng cách 9); khoảng trống: [(-1, 0), (0, 9), (9, 10)]
seat() -> 4 (khoảng trống (0, 9), khoảng cách 4); khoảng trống: [(-1, 0), (0, 4), (4, 9), (9, 10)]
seat() -> 2 (khoảng trống (0, 4), khoảng cách 2); khoảng trống: [(-1, 0), (0, 2), (2, 4), (4, 9), (9, 10)]
leave(4): gộp (2, 4) và (4, 9); khoảng trống: [(-1, 0), (0, 2), (2, 9), (9, 10)]
seat() -> 5 (khoảng trống (2, 9), khoảng cách 3); khoảng trống: [(-1, 0), (0, 2), (2, 5), (5, 9), (9, 10)]


## 5. Cách 1 — Brute Force: duyệt từng ghế

**Ý tưởng.** Lưu tập ghế có người. `seat()`: nếu không có ai thì `0`; ngược lại duyệt **mọi ghế trống** `i` từ `0` tới `n - 1`, tính khoảng cách tới người gần nhất, lấy ghế có khoảng cách lớn nhất (hòa: ghế nhỏ nhất vì duyệt từ trái và cập nhật bằng `>` ngặt). `leave(p)`: xóa `p` khỏi tập.

Đây là định nghĩa trực tiếp nên cũng là **oracle** (chỉ dùng cho phòng nhỏ).

In [8]:
class ExamRoomBrute:
    def __init__(self, n: int):
        self.n = n
        self.occupied = set()

    def seat(self) -> int:
        if not self.occupied:
            self.occupied.add(0)
            return 0
        best_d, best_i = -1, -1
        for i in range(self.n):
            if i in self.occupied:
                continue
            d = min(abs(i - j) for j in self.occupied)
            if d > best_d:                         # '>' ngặt: hòa giữ ghế nhỏ hơn (duyệt từ trái)
                best_d, best_i = d, i
        self.occupied.add(best_i)
        return best_i

    def leave(self, p: int) -> None:
        self.occupied.remove(p)

ExamRoomBrute.__name__ = "ExamRoom"
brute_cls = ExamRoomBrute
res = run_ops(brute_cls, example_ops, example_args)
print(res)
assert res == example_expected

[None, 0, 9, 4, 2, None, 5]


**Độ phức tạp và nút thắt.** `seat()` tốn `O(n * k)` với `k` là số người đang ngồi (mỗi ghế trống tính khoảng cách tới mọi người); với `n = 10^9` là **không thể** (và còn không lưu nổi cả hàng). Nút thắt: ta xét **mọi ghế**, trong khi ghế tối ưu luôn nằm ở **giữa một khoảng trống** hoặc ở **hai đầu hàng**.

In [9]:
for n in (100, 1000, 3000):
    room = ExamRoomBrute(n)
    t0 = time.perf_counter()
    for _ in range(20):
        room.seat()
    print(f"n={n:5}: 20 lần seat() mất {1000 * (time.perf_counter() - t0):9.1f} ms")
print("với n = 10^9 phải duyệt 10^9 ghế mỗi lần: không khả thi")

n=  100: 20 lần seat() mất       1.3 ms
n= 1000: 20 lần seat() mất      15.5 ms
n= 3000: 20 lần seat() mất      48.5 ms
với n = 10^9 phải duyệt 10^9 ghế mỗi lần: không khả thi


## 6. Ý tưởng thuật toán / bước đột phá

**Chỉ giữ người đang ngồi.** Ghế tối ưu luôn thuộc tập `{0, n - 1} ∪ {trung điểm các khoảng trống giữa hai người}` (chứng minh ở mục 10). Vậy `seat()` chỉ cần xét **các khoảng trống**. Có ba mức tối ưu:

| Cách | Ý tưởng | `seat()` | `leave()` |
|---|---|---|---|
| 2 | Danh sách người ngồi (sắp xếp); duyệt các khoảng trống | `O(k)` | `O(k)` (xóa khỏi list) |
| 3 | **Đống ưu tiên** các khoảng trống `(-d, l, r)` + **xóa lười** | `O(log k)` | `O(log k)` |
| 4 | **Tập có thứ tự** các khoảng trống (danh sách sắp xếp + `bisect`), xóa chính xác | `O(log k + k * memmove)` | cùng |

Trong đó `k` là số người đang ngồi (tối đa `10^4`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt các khoảng trống giữa người đang ngồi

**Giải thích từng đoạn.**

- `self.occ`: danh sách **đã sắp xếp** các ghế có người.
- `seat()`:
  - rỗng -> ghế 0.
  - ngược lại xét lần lượt các ứng viên theo thứ tự ghế tăng dần: đầu hàng (khoảng cách `occ[0]`, ghế 0), các khoảng giữa (khoảng cách `(b - a) // 2`, ghế `(a + b) // 2`), cuối hàng (khoảng cách `n - 1 - occ[-1]`, ghế `n - 1`); chỉ cập nhật khi khoảng cách **lớn hơn ngặt** (giữ ghế nhỏ nhất khi hòa).
  - `insort` ghế vào `occ`.
- `leave(p)`: `del occ[bisect_left(occ, p)]`.

In [10]:
class ExamRoomScan:
    def __init__(self, n: int):
        self.n = n
        self.occ: List[int] = []

    def seat(self) -> int:
        if not self.occ:
            self.occ.append(0)
            return 0
        best_d, best_pos = self.occ[0], 0                    # đầu hàng (có thể = 0 nếu ghế 0 có người)
        for a, b in zip(self.occ, self.occ[1:]):
            d = (b - a) // 2
            if d > best_d:
                best_d, best_pos = d, (a + b) // 2
        d = self.n - 1 - self.occ[-1]                        # cuối hàng
        if d > best_d:
            best_d, best_pos = d, self.n - 1
        insort(self.occ, best_pos)
        return best_pos

    def leave(self, p: int) -> None:
        del self.occ[bisect_left(self.occ, p)]

ExamRoomScan.__name__ = "ExamRoom"
scan_cls = ExamRoomScan
res = run_ops(scan_cls, example_ops, example_args)
print(res)
assert res == example_expected

[None, 0, 9, 4, 2, None, 5]


**Code nộp LeetCode (Cách 2):**

In [11]:
from bisect import bisect_left, insort

class ExamRoom:

    def __init__(self, n: int):
        self.n = n
        self.occ = []

    def seat(self) -> int:
        if not self.occ:
            self.occ.append(0)
            return 0
        best_d, best_pos = self.occ[0], 0
        for a, b in zip(self.occ, self.occ[1:]):
            d = (b - a) // 2
            if d > best_d:
                best_d, best_pos = d, (a + b) // 2
        d = self.n - 1 - self.occ[-1]
        if d > best_d:
            best_d, best_pos = d, self.n - 1
        insort(self.occ, best_pos)
        return best_pos

    def leave(self, p: int) -> None:
        del self.occ[bisect_left(self.occ, p)]


# Your ExamRoom object will be instantiated and called as such:
# obj = ExamRoom(n)
# param_1 = obj.seat()
# obj.leave(p)

room_scan_submit = ExamRoom
assert run_ops(room_scan_submit, example_ops, example_args) == example_expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp (Cách 2):** `seat()` `O(k)` (duyệt các khoảng trống) và `insort` `O(k)`; `leave()` `O(k)` (xóa khỏi list); bộ nhớ `O(k)`. Với `k <= 10^4` và `10^4` lần gọi: cỡ `10^8` phép đơn giản trong tình huống xấu nhất (thực tế nhỏ hơn).

---

### 7.2. Cách 3 — Đống ưu tiên các khoảng trống (xóa lười)

**Ý tưởng khác.** Giữ một **đống** các khoảng trống `(-d, l, r)` để lấy khoảng trống tốt nhất trong `O(log k)`. Việc `leave` làm **biến mất** hai khoảng trống và **sinh** một khoảng trống mới; ta không xóa được phần tử bất kỳ khỏi đống nên dùng **xóa lười**: một mục `(−d, l, r)` còn **hợp lệ** khi và chỉ khi `l` vẫn có mặt và láng giềng phải của `l` vẫn là `r`.

**Giải thích từng đoạn.**

- `prev`, `nxt`: láng giềng của mỗi người ngồi và của hai ký tự gác `-1`, `n`.
- `seat()`: lấy ra từ đống đến khi gặp mục hợp lệ (`nxt.get(l) == r`); tính ghế `pos` bằng `gap_info`; **tách** `(l, r)` thành `(l, pos)` và `(pos, r)`: cập nhật `nxt`, `prev`, đẩy hai khoảng trống mới vào đống.
- `leave(p)`: `l = prev.pop(p)`, `r = nxt.pop(p)`; nối `nxt[l] = r`, `prev[r] = l`; đẩy `(l, r)` vào đống. Hai mục cũ `(l, p)` và `(p, r)` trở thành lỗi thời (vì `nxt[l] != p`... xem kiểm tra bên dưới).
- Phòng trống: khoảng trống `(-1, n)` có `nxt[-1] = n`.

In [12]:
class ExamRoomHeap:
    def __init__(self, n: int):
        self.n = n
        self.nxt = {-1: n}
        self.prev = {n: -1}
        self.heap = []
        self._push(-1, n)

    def _dist(self, l: int, r: int) -> int:
        return gap_info(l, r, self.n)[0]

    def _push(self, l: int, r: int) -> None:
        heapq.heappush(self.heap, (-self._dist(l, r), l, r))

    def seat(self) -> int:
        while True:
            neg_d, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:                 # còn hợp lệ: l vẫn có mặt và láng giềng phải vẫn là r
                break
        pos = gap_info(l, r, self.n)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev.pop(p), self.nxt.pop(p)
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)

ExamRoomHeap.__name__ = "ExamRoom"
heap_cls = ExamRoomHeap
res = run_ops(heap_cls, example_ops, example_args)
print(res)
assert res == example_expected

[None, 0, 9, 4, 2, None, 5]


**Code nộp LeetCode (Cách 3):**

In [13]:
import heapq

class ExamRoom:

    def __init__(self, n: int):
        self.n = n
        self.nxt = {-1: n}            # láng giềng phải (gồm cả ký tự gác -1)
        self.prev = {n: -1}           # láng giềng trái (gồm cả ký tự gác n)
        self.heap = []
        self._push(-1, n)

    def _best(self, l: int, r: int):
        n = self.n
        if l == -1:
            return (n, 0) if r == n else (r, 0)
        if r == n:
            return (n - 1 - l, n - 1)
        return ((r - l) // 2, (l + r) // 2)

    def _push(self, l: int, r: int) -> None:
        heapq.heappush(self.heap, (-self._best(l, r)[0], l, r))

    def seat(self) -> int:
        while True:
            _, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:
                break
        pos = self._best(l, r)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev.pop(p), self.nxt.pop(p)
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)


# Your ExamRoom object will be instantiated and called as such:
# obj = ExamRoom(n)
# param_1 = obj.seat()
# obj.leave(p)

room_heap_submit = ExamRoom
assert run_ops(room_heap_submit, example_ops, example_args) == example_expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp (Cách 3):** mỗi `seat`/`leave` đẩy tối đa 2 mục và lấy ra một số mục lỗi thời; tổng số mục đẩy `O(số thao tác)` nên **chi phí khấu hao** `O(log k)` mỗi thao tác. Bộ nhớ `O(số thao tác)` cho đống (mục lỗi thời chưa bị lấy ra) cộng `O(k)` cho các dict.

**Về tính hợp lệ khi xóa lười.** Mục `(−d, l, r)` bị lỗi thời khi `l` rời đi (`nxt.pop(l)` làm `nxt.get(l)` là `None`) hoặc khi một người mới ngồi vào giữa `l` và `r` (`nxt[l]` đổi thành người đó). Một mục có thể "sống lại" nếu người ở giữa rời đi (`nxt[l] = r` trở lại) và ta cũng đẩy một mục mới `(l, r)`; khi đó hai mục giống hệt nhau: mục đầu tiên được lấy ra sẽ dùng khoảng trống này, mục còn lại thành lỗi thời ngay sau đó (vì `nxt[l]` đã đổi).

---

### 7.3. Cách 4 — Tập có thứ tự (danh sách sắp xếp + `bisect`)

**Ý tưởng khác.** Mô phỏng `SortedList` của thư viện `sortedcontainers` (không có trong thư viện chuẩn) bằng một **list đã sắp xếp** các khoảng trống với khóa `(-d, l, r)`: `insort` để thêm, `bisect_left` + `del` để xóa **chính xác** (không còn mục lỗi thời). Khoảng trống tốt nhất luôn là phần tử đầu `gaps[0]`.

**Giải thích từng đoạn.**

- `add(l, r)`: `insort(self.gaps, (-d, l, r))`; ghi `nxt[l] = r`, `prev[r] = l`.
- `remove(l, r)`: tìm vị trí bằng `bisect_left(self.gaps, (-d, l, r))` rồi `del`.
- `seat()`: `(_, l, r) = gaps[0]`, tách thành hai khoảng trống.
- `leave(p)`: xóa `(l, p)` và `(p, r)`, thêm `(l, r)`.

In [14]:
class ExamRoomOrdered:
    def __init__(self, n: int):
        self.n = n
        self.gaps: List[tuple] = []
        self.nxt = {}
        self.prev = {}
        self._add(-1, self.n)

    def _key(self, l: int, r: int) -> tuple:
        return (-gap_info(l, r, self.n)[0], l, r)

    def _add(self, l: int, r: int) -> None:
        insort(self.gaps, self._key(l, r))
        self.nxt[l] = r
        self.prev[r] = l

    def _remove(self, l: int, r: int) -> None:
        del self.gaps[bisect_left(self.gaps, self._key(l, r))]
        del self.nxt[l]
        del self.prev[r]

    def seat(self) -> int:
        _, l, r = self.gaps[0]
        pos = gap_info(l, r, self.n)[1]
        self._remove(l, r)
        self._add(l, pos)
        self._add(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev[p], self.nxt[p]
        self._remove(l, p)
        self._remove(p, r)
        self._add(l, r)

ExamRoomOrdered.__name__ = "ExamRoom"
ordered_cls = ExamRoomOrdered
res = run_ops(ordered_cls, example_ops, example_args)
print(res)
assert res == example_expected

[None, 0, 9, 4, 2, None, 5]


**Code nộp LeetCode (Cách 4):**

In [15]:
from bisect import bisect_left, insort

class ExamRoom:

    def __init__(self, n: int):
        self.n = n
        self.gaps = []                # danh sách đã sắp xếp các (-khoảng_cách, l, r)
        self.nxt = {}
        self.prev = {}
        self._add(-1, n)

    def _best(self, l: int, r: int):
        n = self.n
        if l == -1:
            return (n, 0) if r == n else (r, 0)
        if r == n:
            return (n - 1 - l, n - 1)
        return ((r - l) // 2, (l + r) // 2)

    def _key(self, l: int, r: int):
        return (-self._best(l, r)[0], l, r)

    def _add(self, l: int, r: int) -> None:
        insort(self.gaps, self._key(l, r))
        self.nxt[l] = r
        self.prev[r] = l

    def _remove(self, l: int, r: int) -> None:
        del self.gaps[bisect_left(self.gaps, self._key(l, r))]
        del self.nxt[l]
        del self.prev[r]

    def seat(self) -> int:
        _, l, r = self.gaps[0]
        pos = self._best(l, r)[1]
        self._remove(l, r)
        self._add(l, pos)
        self._add(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev[p], self.nxt[p]
        self._remove(l, p)
        self._remove(p, r)
        self._add(l, r)


# Your ExamRoom object will be instantiated and called as such:
# obj = ExamRoom(n)
# param_1 = obj.seat()
# obj.leave(p)

room_ordered_submit = ExamRoom
assert run_ops(room_ordered_submit, example_ops, example_args) == example_expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp (Cách 4):** `seat`/`leave` mỗi cái 2-3 lần `bisect` `O(log k)` và `insort`/`del` `O(k)` dịch chuyển phần tử (thực tế rất nhanh với `k <= 10^4`); bộ nhớ `O(k)`. Ưu điểm: không có mục lỗi thời. Với `SortedList` thật (`sortedcontainers`) thao tác là `O(log k)` khấu hao.

## 8. TRACE từng bước

### 8.1. Cách 3 (đống, xóa lười) trên ví dụ 1

Hàm dưới in sau mỗi thao tác: đống, các láng giềng và những mục lỗi thời mà `seat()` đã bỏ qua.

In [16]:
class TracedHeap(ExamRoomHeap):
    def seat(self) -> int:
        skipped = []
        while True:
            neg_d, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:
                break
            skipped.append((-neg_d, l, r))
        pos = gap_info(l, r, self.n)[1]
        print(f"  seat(): lấy khoảng trống ({l}, {r}) khoảng cách {-neg_d} -> ghế {pos}" + (f"; bỏ qua mục lỗi thời {skipped}" if skipped else ""))
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev.pop(p), self.nxt.pop(p)
        print(f"  leave({p}): gộp ({l}, {p}) và ({p}, {r}) thành ({l}, {r})")
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)

room = TracedHeap(10)
out = []
for kind, arg in [("seat", None)] * 4 + [("leave", 4), ("seat", None), ("seat", None)]:
    out.append(room.seat() if kind == "seat" else room.leave(arg))
    live = sorted((l, r) for l, r in room.nxt.items())
    print(f"     láng giềng phải: {dict(sorted(room.nxt.items()))}; kích thước đống = {len(room.heap)}")
print("kết quả:", out)

  seat(): lấy khoảng trống (-1, 10) khoảng cách 10 -> ghế 0
     láng giềng phải: {-1: 0, 0: 10}; kích thước đống = 2
  seat(): lấy khoảng trống (0, 10) khoảng cách 9 -> ghế 9
     láng giềng phải: {-1: 0, 0: 9, 9: 10}; kích thước đống = 3
  seat(): lấy khoảng trống (0, 9) khoảng cách 4 -> ghế 4
     láng giềng phải: {-1: 0, 0: 4, 4: 9, 9: 10}; kích thước đống = 4
  seat(): lấy khoảng trống (0, 4) khoảng cách 2 -> ghế 2
     láng giềng phải: {-1: 0, 0: 2, 2: 4, 4: 9, 9: 10}; kích thước đống = 5
  leave(4): gộp (2, 4) và (4, 9) thành (2, 9)
     láng giềng phải: {-1: 0, 0: 2, 2: 9, 9: 10}; kích thước đống = 6
  seat(): lấy khoảng trống (2, 9) khoảng cách 3 -> ghế 5
     láng giềng phải: {-1: 0, 0: 2, 2: 5, 5: 9, 9: 10}; kích thước đống = 7
  seat(): lấy khoảng trống (5, 9) khoảng cách 2 -> ghế 7; bỏ qua mục lỗi thời [(2, 4, 9)]
     láng giềng phải: {-1: 0, 0: 2, 2: 5, 5: 7, 7: 9, 9: 10}; kích thước đống = 7
kết quả: [0, 9, 4, 2, None, 5, 7]


### 8.2. Cách 2: duyệt các khoảng trống

Với `occ = [0, 4, 9]` và `n = 10`, các ứng viên: đầu hàng (`occ[0] = 0`, khoảng cách 0), `(0, 4)` cho `2` (ghế 2), `(4, 9)` cho `2` (ghế 6), cuối hàng `10 - 1 - 9 = 0`. Hòa 2: giữ ghế nhỏ nhất (2) vì chỉ cập nhật khi `>` ngặt.

In [17]:
def trace_scan(n, occ):
    print(f"n = {n}, người đang ngồi = {occ}")
    cands = []
    if occ:
        cands.append((occ[0], 0, "đầu hàng"))
        for a, b in zip(occ, occ[1:]):
            cands.append(((b - a) // 2, (a + b) // 2, f"giữa {a} và {b}"))
        cands.append((n - 1 - occ[-1], n - 1, "cuối hàng"))
    best = None
    for d, pos, name in cands:
        flag = ""
        if best is None or d > best[0]:
            best = (d, pos)
            flag = "  <- tốt nhất hiện tại"
        print(f"  {name:18} khoảng cách {d}, ghế {pos}{flag}")
    return best[1]

assert trace_scan(10, [0, 4, 9]) == 2
print()
assert trace_scan(10, [2, 9]) == 5

n = 10, người đang ngồi = [0, 4, 9]
  đầu hàng           khoảng cách 0, ghế 0  <- tốt nhất hiện tại
  giữa 0 và 4        khoảng cách 2, ghế 2  <- tốt nhất hiện tại
  giữa 4 và 9        khoảng cách 2, ghế 6
  cuối hàng          khoảng cách 0, ghế 9

n = 10, người đang ngồi = [2, 9]
  đầu hàng           khoảng cách 2, ghế 0  <- tốt nhất hiện tại
  giữa 2 và 9        khoảng cách 3, ghế 5  <- tốt nhất hiện tại
  cuối hàng          khoảng cách 0, ghế 9


### 8.3. Cách 4: danh sách khoảng trống có thứ tự

Danh sách `gaps` luôn được sắp xếp theo `(-khoảng_cách, l, r)`; phần tử đầu là khoảng trống sẽ chọn.

In [18]:
room = ExamRoomOrdered(10)
for step in range(1, 5):
    pos = room.seat()
    print(f"seat() -> {pos}; gaps (khoảng cách, l, r) = {[(-k[0], k[1], k[2]) for k in room.gaps]}")
room.leave(4)
print("leave(4); gaps =", [(-k[0], k[1], k[2]) for k in room.gaps])
pos = room.seat()
print(f"seat() -> {pos}; gaps =", [(-k[0], k[1], k[2]) for k in room.gaps])
assert pos == 5

seat() -> 0; gaps (khoảng cách, l, r) = [(9, 0, 10), (0, -1, 0)]
seat() -> 9; gaps (khoảng cách, l, r) = [(4, 0, 9), (0, -1, 0), (0, 9, 10)]
seat() -> 4; gaps (khoảng cách, l, r) = [(2, 0, 4), (2, 4, 9), (0, -1, 0), (0, 9, 10)]
seat() -> 2; gaps (khoảng cách, l, r) = [(2, 4, 9), (1, 0, 2), (1, 2, 4), (0, -1, 0), (0, 9, 10)]
leave(4); gaps = [(3, 2, 9), (1, 0, 2), (0, -1, 0), (0, 9, 10)]
seat() -> 5; gaps = [(2, 5, 9), (1, 0, 2), (1, 2, 5), (0, -1, 0), (0, 9, 10)]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (đống + xóa lười): chỉ dùng thư viện chuẩn, `seat`/`leave` khấu hao `O(log k)`.

```python
import heapq

class ExamRoom:

    def __init__(self, n: int):
        self.n = n
        self.nxt = {-1: n}
        self.prev = {n: -1}
        self.heap = []
        self._push(-1, n)

    def _best(self, l: int, r: int):
        n = self.n
        if l == -1:
            return (n, 0) if r == n else (r, 0)
        if r == n:
            return (n - 1 - l, n - 1)
        return ((r - l) // 2, (l + r) // 2)

    def _push(self, l: int, r: int) -> None:
        heapq.heappush(self.heap, (-self._best(l, r)[0], l, r))

    def seat(self) -> int:
        while True:
            _, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:
                break
        pos = self._best(l, r)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev.pop(p), self.nxt.pop(p)
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)
```

(Cell dưới định nghĩa lại đúng bản này để chạy qua bộ `run_ops`.)

In [19]:
import heapq

class ExamRoom:

    def __init__(self, n: int):
        self.n = n
        self.nxt = {-1: n}
        self.prev = {n: -1}
        self.heap = []
        self._push(-1, n)

    def _best(self, l: int, r: int):
        n = self.n
        if l == -1:
            return (n, 0) if r == n else (r, 0)
        if r == n:
            return (n - 1 - l, n - 1)
        return ((r - l) // 2, (l + r) // 2)

    def _push(self, l: int, r: int) -> None:
        heapq.heappush(self.heap, (-self._best(l, r)[0], l, r))

    def seat(self) -> int:
        while True:
            _, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:
                break
        pos = self._best(l, r)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev.pop(p), self.nxt.pop(p)
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)

room_final = ExamRoom
res = run_ops(room_final, example_ops, example_args)
print(res)
assert res == example_expected

[None, 0, 9, 4, 2, None, 5]


## 10. Vì sao đúng

**(a) Ghế tối ưu nằm ở trung điểm khoảng trống hoặc ở hai đầu.** Mỗi ghế trống thuộc đúng một khoảng trống `(l, r)` (giữa hai người liên tiếp hoặc giữa một người và đầu hàng). Với ghế `i` trong `(l, r)` giữa hai người, khoảng cách tới người gần nhất là `min(i - l, r - i)`, lớn nhất là `(r - l) // 2` tại `i = (l + r) // 2` (và `i = (l + r + 1) // 2` nếu `r - l` lẻ; ghế **nhỏ nhất** đạt max là `(l + r) // 2`). Với khoảng trống đầu hàng `(-1, r)` chỉ có người bên phải: khoảng cách `r - i` lớn nhất tại `i = 0` (giá trị `r`); khoảng trống cuối hàng `(l, n)` tương tự tại `i = n - 1` (giá trị `n - 1 - l`). Do đó mỗi khoảng trống có **một ứng viên** và khoảng cách của nó đã tính ở `gap_info`.

**(b) Phá hòa.** Hai khoảng trống `A = (l_1, r_1)`, `B = (l_2, r_2)` cùng khoảng cách tối đa, `l_1 < l_2`. Ghế của `A` nhỏ hơn ghế của `B`: các khoảng trống không giao nhau theo thứ tự trái -> phải (`r_1 <= l_2`), nên mọi ghế của `A` nằm trước `l_2` (ghế trong `A` nằm trong `(l_1, r_1)` và `r_1 <= l_2 <` ghế của `B`); ghế đầu hàng là `0` (nhỏ nhất) ứng với `l = -1` (nhỏ nhất); ghế cuối hàng là `n - 1` (lớn nhất) ứng với khoảng trống có `l` lớn nhất. Vậy sắp xếp theo `(−khoảng_cách, l)` cho đúng ghế nhỏ nhất khi hòa.

**(c) `leave` gộp khoảng trống.** Khi người ở `p` rời đi, các ghế của `(l, p)`, `p` và `(p, r)` đều trở thành trống và liên tục: một khoảng trống `(l, r)`.

**(d) Cách 3: xóa lười đúng.** Một mục `(−d, l, r)` đại diện cho khoảng trống **đang tồn tại** khi và chỉ khi `l` có mặt (là ký tự gác hoặc người đang ngồi) và `nxt[l] == r`. Mọi khoảng trống đang tồn tại có ít nhất một mục hợp lệ trong đống (được đẩy ngay khi sinh ra), mọi mục lỗi thời bị bỏ qua khi lấy ra. Mục hợp lệ **tốt nhất** (nhỏ nhất theo `(−d, l)`) là khoảng trống tối ưu, vì khoảng cách `d` của một khoảng trống `(l, r)` chỉ phụ thuộc `(l, r)` và `n` (không đổi theo thời gian).

**(e) Cách 4** giữ chính xác tập khoảng trống nên `gaps[0]` là khoảng trống tối ưu.

**Kiểm chứng bằng máy:** chạy nhiều **chuỗi lệnh ngẫu nhiên** (nhiều `n`, tới `10^9`) lên cả bốn cách và so sánh **mọi kết quả** của `seat()` với oracle duyệt từng ghế (phòng nhỏ) hoặc với Cách 2 (phòng lớn, đã được oracle kiểm).

In [20]:
def gen_ops(rng, n, steps, p_seat=0.6, ref_cls=None):
    """Sinh chuỗi lệnh hợp lệ cho phòng n ghế; dùng một cài đặt tham chiếu (có thuộc tính occ) để biết ai đang ngồi."""
    ref = (ref_cls or ExamRoomScan)(n)
    ops, args = ["ExamRoom"], [[n]]
    for _ in range(steps):
        full = len(ref.occ) == n
        if ref.occ and (full or rng.random() > p_seat):
            p = rng.choice(ref.occ)
            ref.leave(p)
            ops.append("leave"); args.append([p])
        else:
            ref.seat()
            ops.append("seat"); args.append([])
    return ops, args

implementations = {
    "scan": scan_cls,
    "heap": heap_cls,
    "ordered": ordered_cls,
    "submit_scan": room_scan_submit,
    "submit_heap": room_heap_submit,
    "submit_ordered": room_ordered_submit,
    "final": room_final,
}

rng = random.Random(855)
checked = 0
for n in list(range(1, 13)) + [20, 33, 50]:
    for _ in range(25):
        ops, args = gen_ops(rng, n, rng.randint(1, 60))
        ref = run_ops(brute_cls, ops, args)                         # oracle: duyệt từng ghế
        for name, cls in implementations.items():
            assert run_ops(cls, ops, args) == ref, (name, n, ops, args)
        checked += 1
print(f"{checked} chuỗi lệnh ngẫu nhiên trên phòng nhỏ (n = 1..50) x {len(implementations)} cài đặt khớp oracle duyệt từng ghế")

375 chuỗi lệnh ngẫu nhiên trên phòng nhỏ (n = 1..50) x 7 cài đặt khớp oracle duyệt từng ghế


## 11. Time / Space Complexity

Gọi `k` là số người đang ngồi (`<= min(n, 10^4)`), `m` là số lệnh (`<= 10^4`).

| Cách | `seat()` | `leave()` | Space |
|---|---|---|---|
| 1. Duyệt từng ghế | `O(n k)` | `O(1)` | `O(k)` |
| 2. Duyệt khoảng trống | `O(k)` | `O(k)` | `O(k)` |
| 3. Đống + xóa lười | `O(log m)` khấu hao | `O(log m)` | `O(m)` |
| 4. Danh sách sắp xếp + `bisect` | `O(log k + k)` (dịch phần tử, rất nhanh) | cùng | `O(k)` |
| (SortedList thật) | `O(log k)` | `O(log k)` | `O(k)` |

Đo thử với `n = 10^9` và `10^4` lệnh (bỏ qua Cách 1):

In [21]:
rng = random.Random(8550)
n = 10 ** 9
ops, args = gen_ops(rng, n, 10000, p_seat=0.8)
print("số lệnh:", len(ops) - 1, "| số lệnh seat:", ops.count("seat"), "| leave:", ops.count("leave"))
ref = None
for name in ["heap", "ordered", "scan"]:
    cls = implementations[name]
    t0 = time.perf_counter()
    res = run_ops(cls, ops, args)
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:8} {len(ops) - 1} lệnh, n = 10^9: {dt:9.1f} ms")

số lệnh: 10000 | số lệnh seat: 7923 | leave: 2077
heap     10000 lệnh, n = 10^9:      21.7 ms
ordered  10000 lệnh, n = 10^9:      45.3 ms


scan     10000 lệnh, n = 10^9:    1790.2 ms


## 12. Bộ test

Gồm: ví dụ của đề (qua `run_ops`), edge case, phản ví dụ cho các cách sai (tìm bằng chuỗi lệnh ngẫu nhiên có seed), chuỗi lệnh ngẫu nhiên có seed (nhiều `n`, tới `10^9`), nhiều đối tượng độc lập.

In [22]:
# 12.1 ví dụ đề
for name, cls in implementations.items():
    assert run_ops(cls, example_ops, example_args) == example_expected, name
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
cases = {
    "một ghế": (["ExamRoom", "seat", "leave", "seat"], [[1], [], [0], []], [None, 0, None, 0]),
    "hai ghế": (["ExamRoom", "seat", "seat"], [[2], [], []], [None, 0, 1]),
    "ba ghế": (["ExamRoom", "seat", "seat", "seat"], [[3], [], [], []], [None, 0, 2, 1]),
    "phòng lớn n = 10^9": (["ExamRoom", "seat", "seat", "seat"], [[10 ** 9], [], [], []], [None, 0, 10 ** 9 - 1, (10 ** 9 - 1) // 2]),
    "rời rồi ngồi lại ghế 0": (["ExamRoom", "seat", "seat", "leave", "seat"], [[5], [], [], [0], []], [None, 0, 4, None, 0]),
    "hòa giữa hai khoảng trống": (["ExamRoom", "seat", "seat", "seat", "seat"], [[9], [], [], [], []], [None, 0, 8, 4, 2]),
    "người ở giữa phòng rồi rời": (["ExamRoom", "seat", "seat", "seat", "leave", "seat"], [[7], [], [], [], [6], []], [None, 0, 6, 3, None, 6]),
}
for title, (ops, args, expected) in cases.items():
    if args[0][0] <= 100:                                           # oracle duyệt từng ghế chỉ dùng cho phòng nhỏ
        assert run_ops(brute_cls, ops, args) == expected, ("oracle khác kỳ vọng", title, run_ops(brute_cls, ops, args))
    for name, cls in implementations.items():
        assert run_ops(cls, ops, args) == expected, (title, name, run_ops(cls, ops, args), expected)
print("12.2 edge case: OK", len(cases), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 7 ca


In [23]:
# 12.3 phản ví dụ: các cài đặt sai, và tìm một chuỗi lệnh ngắn lộ lỗi bằng tìm kiếm có seed
class WrongCeilMid:                 # trung điểm làm tròn lên: không phải ghế nhỏ nhất khi hòa
    def __init__(self, n):
        self.n, self.occ = n, []
    def seat(self):
        if not self.occ:
            self.occ.append(0); return 0
        best_d, best_pos = self.occ[0], 0
        for a, b in zip(self.occ, self.occ[1:]):
            d = (b - a) // 2
            if d > best_d:
                best_d, best_pos = d, (a + b + 1) // 2
        d = self.n - 1 - self.occ[-1]
        if d > best_d:
            best_d, best_pos = d, self.n - 1
        insort(self.occ, best_pos)
        return best_pos
    def leave(self, p):
        self.occ.remove(p)

class WrongHalfEnds:                # chia đôi cả hai đầu hàng
    def __init__(self, n):
        self.n, self.occ = n, []
    def seat(self):
        if not self.occ:
            self.occ.append(0); return 0
        cands = [(self.occ[0] // 2, 0 if self.occ[0] == 0 else self.occ[0] // 2)]
        for a, b in zip(self.occ, self.occ[1:]):
            cands.append(((b - a) // 2, (a + b) // 2))
        cands.append(((self.n - 1 - self.occ[-1]) // 2, (self.n - 1 + self.occ[-1]) // 2))
        best = max(cands, key=lambda c: c[0])
        insort(self.occ, best[1])
        return best[1]
    def leave(self, p):
        self.occ.remove(p)

class WrongTieHigh:                 # hòa -> chọn ghế LỚN hơn (cập nhật bằng >=)
    def __init__(self, n):
        self.n, self.occ = n, []
    def seat(self):
        if not self.occ:
            self.occ.append(0); return 0
        best_d, best_pos = self.occ[0], 0
        for a, b in zip(self.occ, self.occ[1:]):
            d = (b - a) // 2
            if d >= best_d and d > 0:
                best_d, best_pos = d, (a + b) // 2
        d = self.n - 1 - self.occ[-1]
        if d >= best_d and d > 0:
            best_d, best_pos = d, self.n - 1
        insort(self.occ, best_pos)
        return best_pos
    def leave(self, p):
        self.occ.remove(p)

class WrongNoValidate(ExamRoomHeap):     # đống xóa lười nhưng KHÔNG kiểm tra mục lỗi thời
    def seat(self):
        neg_d, l, r = heapq.heappop(self.heap)
        pos = gap_info(l, r, self.n)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

class WrongMinHeapPositive(ExamRoomHeap):   # đống nhỏ nhất với khoảng cách DƯƠNG (lấy khoảng trống nhỏ nhất)
    def _push(self, l, r):
        heapq.heappush(self.heap, (self._dist(l, r), l, r))
    def seat(self):
        while True:
            d, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:
                break
        pos = gap_info(l, r, self.n)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

def _renamed(cls):
    cls.__name__ = "ExamRoom"
    return cls

def find_counterexample(wrong_cls, seed=1):
    rng = random.Random(seed)
    for steps in range(1, 40):
        for _ in range(400):
            n = rng.randint(2, 14)
            ops, args = gen_ops(rng, n, steps)
            try:
                got = run_ops(_renamed(wrong_cls), ops, args)
            except Exception as ex:
                got = ["LỖI", repr(ex)]
            if got != run_ops(brute_cls, ops, args):
                return n, ops, args, got
    return None

wrongs = [
    ("trung điểm làm tròn lên", WrongCeilMid),
    ("chia đôi cả hai đầu hàng", WrongHalfEnds),
    ("hòa chọn ghế lớn hơn", WrongTieHigh),
    ("đống không kiểm tra mục lỗi thời", WrongNoValidate),
    ("đống nhỏ nhất với khoảng cách dương", WrongMinHeapPositive),
]
for title, cls in wrongs:
    found = find_counterexample(cls)
    assert found is not None, title
    n, ops, args, got = found
    seq = [("seat()" if o == "seat" else f"leave({a[0]})") for o, a in zip(ops[1:], args[1:])]
    print(f"{title:38} n={n}: {seq}")
    print(f"{'':38} sai  = {got[1:]}")
    print(f"{'':38} đúng = {run_ops(brute_cls, ops, args)[1:]}")
print("12.3 phản ví dụ: OK (mỗi cài đặt sai đều có chuỗi lệnh lộ lỗi)")

trung điểm làm tròn lên                n=8: ['seat()', 'seat()', 'seat()']
                                       sai  = [0, 7, 4]
                                       đúng = [0, 7, 3]
chia đôi cả hai đầu hàng               n=11: ['seat()', 'seat()']
                                       sai  = [0, 5]
                                       đúng = [0, 10]
hòa chọn ghế lớn hơn                   n=10: ['seat()', 'seat()', 'seat()', 'seat()']
                                       sai  = [0, 9, 4, 6]
                                       đúng = [0, 9, 4, 2]
đống không kiểm tra mục lỗi thời       n=14: ['seat()', 'leave(0)', 'seat()', 'seat()', 'seat()']
                                       sai  = [0, None, 0, 13, 13]
                                       đúng = [0, None, 0, 13, 6]
đống nhỏ nhất với khoảng cách dương    n=11: ['seat()', 'seat()']
                                       sai  = [0, 0]
                                       đúng = [0, 10]
12.3 phản ví dụ: OK (mỗi cài đặt

In [24]:
# 12.4 ngẫu nhiên lớn: n tới 10^9, chuỗi lệnh dài; các cách khớp nhau (Cách 2 đã được oracle kiểm trên phòng nhỏ)
rng = random.Random(2855)
for _ in range(25):
    n = rng.choice([10 ** 9, 10 ** 6, 12345, 100000, 7])
    ops, args = gen_ops(rng, n, rng.randint(200, 1500), p_seat=rng.choice([0.55, 0.7, 0.9]))
    ref = run_ops(scan_cls, ops, args)
    for name, cls in implementations.items():
        assert run_ops(cls, ops, args) == ref, (name, n)
print("12.4 chuỗi lệnh dài trên phòng lớn: OK")

# 12.5 phòng đầy rồi rời hết: seat luôn hợp lệ và các ghế trả về là hoán vị của 0..n-1
for n in (1, 2, 5, 17, 64):
    for name, cls in implementations.items():
        room = cls(n)
        got = [room.seat() for _ in range(n)]
        assert sorted(got) == list(range(n)), (name, n, got)
        for p in got[::-1]:
            room.leave(p)
        again = [room.seat() for _ in range(n)]
        assert again == got, (name, n)                  # sau khi rời hết, lặp lại cùng chuỗi chọn ghế
print("12.5 ngồi đầy, rời hết, ngồi lại: OK")

# 12.6 nhiều đối tượng độc lập (kiểm tra không dùng chung trạng thái)
for name, cls in implementations.items():
    a, b = cls(10), cls(10)
    assert a.seat() == 0 and b.seat() == 0 and a.seat() == 9
    assert b.seat() == 9 and a.seat() == 4 and b.seat() == 4
print("12.6 nhiều đối tượng độc lập: OK")

12.4 chuỗi lệnh dài trên phòng lớn: OK
12.5 ngồi đầy, rời hết, ngồi lại: OK
12.6 nhiều đối tượng độc lập: OK


## 13. Các lỗi người mới hay gặp

Mỗi lỗi dưới đây đã được **tìm tự động** ở mục 12.3 một chuỗi lệnh lộ lỗi (tìm bằng bộ sinh ngẫu nhiên có seed). Ở đây xem chi tiết một số chuỗi lệnh quen thuộc.

### Lỗi 1 — Trung điểm làm tròn lên

Khi `r - l` lẻ có hai ghế cùng khoảng cách; ghế đúng là ghế **nhỏ hơn** `(l + r) // 2`.

In [25]:
ops = ["ExamRoom", "seat", "seat", "seat", "seat"]
args = [[8], [], [], [], []]
print("đúng :", run_ops(brute_cls, ops, args))
print("sai  :", run_ops(_renamed(WrongCeilMid), ops, args))
assert run_ops(_renamed(WrongCeilMid), ops, args) != run_ops(brute_cls, ops, args)

đúng : [None, 0, 7, 3, 5]
sai  : [None, 0, 7, 4, 2]


Vì sao sai: với người ở `0` và `7`, hai ghế `3` và `4` cùng cách người gần nhất 3 ghế, đề yêu cầu ghế **số nhỏ nhất** (`3`). **Đúng:** `(l + r) // 2`.

### Lỗi 2 — Chia đôi cả hai đầu hàng

Ở đầu hàng chỉ có một người bên cạnh nên khoảng cách là **nguyên độ dài**, không chia đôi.

In [26]:
ops = ["ExamRoom", "seat", "seat", "seat"]
args = [[10], [], [], []]
print("đúng :", run_ops(brute_cls, ops, args))
print("sai  :", run_ops(_renamed(WrongHalfEnds), ops, args))
assert run_ops(_renamed(WrongHalfEnds), ops, args) != run_ops(brute_cls, ops, args)

đúng : [None, 0, 9, 4]
sai  : [None, 0, 4, 2]


Vì sao sai: ghế cuối `n - 1` cách người ở 0 đến `n - 1` ghế; chia đôi làm đầu hàng trông kém giữa. **Đúng:** đầu hàng cho `r` (hoặc `n - 1 - l`), giữa cho `(r - l) // 2`.

### Lỗi 3 — Hòa thì chọn ghế lớn hơn

Cập nhật bằng `>=` thay vì `>` làm ghế sau ghi đè ghế trước khi hòa.

In [27]:
ops = ["ExamRoom", "seat", "seat", "seat", "seat"]
args = [[10], [], [], [], []]
print("đúng :", run_ops(brute_cls, ops, args))
print("sai  :", run_ops(_renamed(WrongTieHigh), ops, args))
assert run_ops(_renamed(WrongTieHigh), ops, args) != run_ops(brute_cls, ops, args)

đúng : [None, 0, 9, 4, 2]
sai  : [None, 0, 9, 4, 6]


Vì sao sai: sau khi `0, 9, 4` đã ngồi, hai khoảng `(0, 4)` và `(4, 9)` cùng cách 2; đề chọn ghế nhỏ nhất (`2`). **Đúng:** `>` ngặt khi duyệt từ trái, hoặc khóa `(-d, l)` trong đống.

### Lỗi 4 — Đống xóa lười nhưng không kiểm tra mục lỗi thời

Lấy thẳng phần tử đầu đống sẽ dùng một khoảng trống đã bị tách hoặc gộp.

In [28]:
ops = ["ExamRoom", "seat", "seat", "seat", "leave", "seat"]
args = [[10], [], [], [], [4], []]
print("đúng :", run_ops(brute_cls, ops, args))
try:
    print("sai  :", run_ops(_renamed(WrongNoValidate), ops, args))
except Exception as ex:
    print("sai  : lỗi", type(ex).__name__, ex)
found = find_counterexample(WrongNoValidate)
assert found is not None and found[3] != run_ops(brute_cls, found[1], found[2])
print("chuỗi lệnh lộ lỗi tìm được:", [("seat()" if o == "seat" else f"leave({a[0]})") for o, a in zip(found[1][1:], found[2][1:])], "n =", found[0])

đúng : [None, 0, 9, 4, None, 4]
sai  : [None, 0, 9, 4, None, 4]
chuỗi lệnh lộ lỗi tìm được: ['seat()', 'leave(0)', 'seat()', 'seat()', 'seat()'] n = 14


Vì sao sai: sau khi `(l, r)` bị tách bởi `seat`, mục `(−d, l, r)` cũ vẫn còn trong đống; nếu lấy ra mà không kiểm tra `nxt[l] == r` ta "ngồi" vào một khoảng trống không còn tồn tại. **Đúng:** `if self.nxt.get(l) == r: break`, ngược lại bỏ qua và lấy tiếp.

### Lỗi 5 — Đống nhỏ nhất với khoảng cách dương

`heapq` là đống **nhỏ nhất**; để lấy khoảng cách **lớn nhất** phải lưu `-d`.

In [29]:
ops = ["ExamRoom", "seat", "seat", "seat", "seat"]
args = [[10], [], [], [], []]
print("đúng :", run_ops(brute_cls, ops, args))
print("sai  :", run_ops(_renamed(WrongMinHeapPositive), ops, args))
assert run_ops(_renamed(WrongMinHeapPositive), ops, args) != run_ops(brute_cls, ops, args)

đúng : [None, 0, 9, 4, 2]
sai  : [None, 0, 0, 0, 0]


Vì sao sai: đống nhỏ nhất trả khoảng trống **nhỏ nhất**, tức học sinh chọn chỗ gần người nhất. **Đúng:** `(-d, l, r)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "tối đa/tối thiểu theo khoảng cách", dữ liệu cực lớn (`n = 10^9`) nhưng số thao tác nhỏ -> lưu **các điểm đang có** và **các khoảng** giữa chúng, không lưu từng phần tử.
- **Mỗi thao tác tách một khoảng thành hai, hoặc gộp hai thành một:** cần biết **láng giềng** của mỗi điểm (dict `prev`/`next` hoặc danh sách liên kết).
- **Cần "lấy khoảng tốt nhất" và xóa phần tử bất kỳ:** đống + **xóa lười** hoặc tập có thứ tự (`SortedList` / `TreeSet`).
- **Phá hòa** nhúng vào khóa so sánh: `(-khoảng_cách, vị_trí)`.
- **Ký tự gác** (`-1`, `n`) xử lý đầu/cuối hàng một cách thống nhất.
- Đây là mở rộng trực tiếp của Maximize Distance to Closest Person (849) từ một lần chọn sang nhiều lần chọn và rời đi.
- Tự dựng bộ chạy chuỗi lệnh (`run_ops`) và oracle đơn giản để kiểm tra bài thiết kế.
- Cùng họ: My Calendar (729, 731), Design Hit Counter, Data Stream as Disjoint Intervals (352), Range Module (715).
- Test nên có: `n = 1`, rời rồi ngồi lại ghế 0, ngồi đầy rồi rời hết, `n = 10^9`.

## 15. Biến thể

1. **Hòa chọn ghế lớn nhất** thay vì nhỏ nhất (bài tập 5): đổi khóa phá hòa và cách tính ghế giữa.
2. **Số học sinh đang ngồi / khoảng cách hiện tại lớn nhất:** thêm các truy vấn phụ.
3. **Hai hàng ghế** hoặc ghế trong vòng tròn: khoảng trống đầu và cuối ghép lại.
4. Bài liên quan: Maximize Distance to Closest Person (849), Seat Reservation Manager (1845).

Dưới đây là code cho biến thể 1 (`ExamRoomHigh`: hòa thì chọn ghế **lớn nhất**) và oracle duyệt từng ghế tương ứng.

In [30]:
class ExamRoomHighBrute:
    def __init__(self, n):
        self.n, self.occupied = n, set()
    def seat(self):
        if not self.occupied:
            self.occupied.add(0)
            return 0
        best_d, best_i = -1, -1
        for i in range(self.n):
            if i in self.occupied:
                continue
            d = min(abs(i - j) for j in self.occupied)
            if d >= best_d:                       # hòa: lấy ghế lớn hơn
                best_d, best_i = d, i
        self.occupied.add(best_i)
        return best_i
    def leave(self, p):
        self.occupied.remove(p)

class ExamRoomHigh:
    """Hòa thì chọn ghế lớn nhất: dùng khóa (-d, -l) và ghế giữa làm tròn lên."""
    def __init__(self, n):
        self.n = n
        self.occ = []
    def seat(self):
        if not self.occ:
            self.occ.append(0)
            return 0
        best_d, best_pos = self.occ[0], 0
        for a, b in zip(self.occ, self.occ[1:]):
            d = (b - a) // 2
            if d >= best_d and d > 0:
                best_d, best_pos = d, (a + b + 1) // 2
        d = self.n - 1 - self.occ[-1]
        if d >= best_d and d > 0:
            best_d, best_pos = d, self.n - 1
        insort(self.occ, best_pos)
        return best_pos
    def leave(self, p):
        del self.occ[bisect_left(self.occ, p)]

ExamRoomHighBrute.__name__ = "ExamRoomHigh"
high_brute = ExamRoomHighBrute
ExamRoomHigh.__name__ = "ExamRoomHigh"
high_fast = ExamRoomHigh

rng = random.Random(15)
for n in list(range(2, 12)) + [20]:
    for _ in range(40):
        ops, args = gen_ops(rng, n, rng.randint(1, 40), ref_cls=ExamRoomHigh)
        ops = ["ExamRoomHigh"] + ops[1:]
        assert run_ops(high_fast, ops, args) == run_ops(high_brute, ops, args), (n, ops, args)
print("Biến thể (hòa chọn ghế lớn nhất): OK")

Biến thể (hòa chọn ghế lớn nhất): OK


## 16. So sánh các cách

| Cách | Ý tưởng | `seat` | `leave` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Duyệt từng ghế | Định nghĩa trực tiếp | `O(n k)` | `O(1)` | `O(k)` | Oracle, `n` nhỏ |
| 2. Duyệt khoảng trống | Danh sách người ngồi sắp xếp | `O(k)` | `O(k)` | `O(k)` | Đơn giản, đủ cho `k <= 10^4` |
| 3. Đống + xóa lười | `(-d, l, r)`, láng giềng bằng dict | `O(log m)` khấu hao | `O(log m)` | `O(m)` | **Cách nộp chính**, chỉ thư viện chuẩn |
| 4. Danh sách khoảng trống có thứ tự | `insort` / `bisect`, xóa chính xác | `O(log k + k)` | cùng | `O(k)` | Muốn không có mục lỗi thời; có `SortedList` thì tốt nhất |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán kết quả `run_ops` của chuỗi lệnh sau với cài đặt `room_final`, rồi chạy để kiểm tra:

```python
ops  = ["ExamRoom", "seat", "seat", "seat", "leave", "seat"]
args = [[6], [], [], [], [0], []]
```

In [31]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess = None      # ví dụ [None, 0, ...]

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 10`, sau các lệnh `seat()`, `seat()`, `seat()`, `leave(9)`: ghi danh sách các **khoảng trống** `(l, r)` (có cả ký tự gác `-1`, `10`) theo thứ tự tăng dần của `l`, và ghế mà lần `seat()` tiếp theo trả về.

In [32]:
# Bài 2: điền kết quả tính tay
hand_gaps = None      # list các tuple (l, r) đã sắp xếp
hand_next_seat = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm tính khoảng cách và ghế tốt nhất của một khoảng trống.

In [33]:
def exercise_3_gap(l: int, r: int, n: int):
    # TODO 1: nếu l == -1 và r == n (phòng trống) trả (n, 0)
    # TODO 2: nếu l == -1 (đầu hàng) trả (r, 0)
    # TODO 3: nếu r == n (cuối hàng) trả (n - 1 - l, n - 1)
    # TODO 4: ngược lại (giữa hai người) trả ((r - l) // 2, (l + r) // 2)
    return None


# assert exercise_3_gap(-1, 10, 10) == (10, 0)
# assert exercise_3_gap(-1, 4, 10) == (4, 0)
# assert exercise_3_gap(0, 10, 10) == (9, 9)
# assert exercise_3_gap(0, 9, 10) == (4, 4)
# assert exercise_3_gap(4, 9, 10) == (2, 6)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `ExerciseRoom4` bằng đống + xóa lười (hoặc danh sách có thứ tự). Dùng `exercise_3_gap` hoặc `gap_info` để tính ghế.

In [34]:
class ExerciseRoom4:
    # TODO: __init__(self, n), seat(self), leave(self, p)
    pass


# ExerciseRoom4.__name__ = "ExamRoom"
# assert run_ops(ExerciseRoom4, example_ops, example_args) == example_expected

### Bài 5 — Mini challenge: hòa thì chọn ghế lớn nhất (khó)

Viết `ExerciseRoom5` giống `ExamRoom` nhưng khi có nhiều ghế cùng khoảng cách tối đa thì chọn ghế có **số lớn nhất** (phòng trống vẫn ngồi ghế `0`). Kiểm tra bằng oracle `high_brute`.

Gợi ý: khóa của khoảng trống `(-d, -l)`; ghế trong khoảng trống giữa là `(l + r + 1) // 2` khi `r - l` lẻ (ghế lớn hơn trong hai ghế tốt nhất); ghế cuối hàng `n - 1`; đầu hàng chỉ có `0`.

In [35]:
class ExerciseRoom5:
    # TODO: __init__(self, n), seat(self), leave(self, p)
    pass


# ExerciseRoom5.__name__ = "ExamRoomHigh"
# rng = random.Random(5)
# for n in range(2, 10):
#     for _ in range(30):
#         ops, args = gen_ops(rng, n, rng.randint(1, 30), ref_cls=ExamRoomHigh)
#         ops = ["ExamRoomHigh"] + ops[1:]
#         assert run_ops(ExerciseRoom5, ops, args) == run_ops(high_brute, ops, args), (n, ops, args)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: ghế đầu tiên là 0; ghế thứ hai là ghế cuối `5`.
- Mức 2: ghế thứ ba là giữa `0` và `5`: `(0 + 5) // 2 = 2`; `leave(0)` làm `(−1, 2)` có khoảng cách 2.
- Mức 3: người ở `2` và `5`; khoảng trống `(-1, 2)`: khoảng cách `2` (ghế 0); `(2, 5)`: `1`; `(5, 6)`: 0. Kết quả `[None, 0, 5, 2, None, 0]`.

**Bài 2**
- Mức 1: `seat()` ba lần cho `0, 9, 4`; `leave(9)` gộp `(4, 9)` và `(9, 10)`.
- Mức 2: người còn lại: `0` và `4`; khoảng trống: `(-1, 0)`, `(0, 4)`, `(4, 10)`.
- Mức 3: `(4, 10)` cuối hàng: khoảng cách `10 - 1 - 4 = 5` (ghế 9) là lớn nhất; `hand_next_seat = 9`.

**Bài 3**
- Mức 1: phòng trống khi `l == -1 and r == n`.
- Mức 2: đầu hàng `(r - l - 1, 0)` với `l = -1` là `(r, 0)`.
- Mức 3: giữa: `((r - l) // 2, (l + r) // 2)`; cuối hàng: `(n - 1 - l, n - 1)`.

**Bài 4**
- Mức 1: `self.nxt = {-1: n}`, `self.prev = {n: -1}`, đống chứa `(-d, l, r)` ban đầu `(−n, −1, n)`.
- Mức 2: `seat`: lấy mục cho tới khi `self.nxt.get(l) == r`, tách thành `(l, pos)` và `(pos, r)`.
- Mức 3: `leave(p)`: `l, r = prev.pop(p), nxt.pop(p)`; `nxt[l] = r`; `prev[r] = l`; đẩy `(l, r)`.

**Bài 5**
- Mức 1: duyệt khoảng trống từ trái sang phải, cập nhật khi `d >= best_d` (và `d > 0`).
- Mức 2: ghế giữa làm tròn lên: `(a + b + 1) // 2`; khi `b - a` chẵn thì bằng `(a + b) // 2` (ghế duy nhất).
- Mức 3: kiểm chứng bằng `high_brute` trên chuỗi lệnh ngẫu nhiên.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [36]:
ops1 = ["ExamRoom", "seat", "seat", "seat", "leave", "seat"]
args1 = [[6], [], [], [], [0], []]
out1 = run_ops(room_final, ops1, args1)
print(out1)
assert out1 == [None, 0, 5, 2, None, 0]
assert run_ops(brute_cls, ops1, args1) == out1

[None, 0, 5, 2, None, 0]


### Đáp án Bài 2

Sau `seat()` ×3 (`0`, `9`, `4`) và `leave(9)`: người ở `0` và `4`; khoảng trống `(-1, 0)`, `(0, 4)`, `(4, 10)`. Lần `seat()` tiếp theo: `(4, 10)` cuối hàng cho khoảng cách `5` (ghế `9`), lớn hơn `(0, 4)` (khoảng cách 2).

In [37]:
hand_gaps_solution = [(-1, 0), (0, 4), (4, 10)]
hand_next_seat_solution = 9
room = ExamRoomOrdered(10)
for _ in range(3):
    room.seat()
room.leave(9)
assert sorted((l, r) for l, r in room.nxt.items()) == hand_gaps_solution
assert room.seat() == hand_next_seat_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [38]:
def exercise_3_gap_solution(l: int, r: int, n: int):
    if l == -1 and r == n:
        return (n, 0)
    if l == -1:
        return (r, 0)
    if r == n:
        return (n - 1 - l, n - 1)
    return ((r - l) // 2, (l + r) // 2)

assert exercise_3_gap_solution(-1, 10, 10) == (10, 0)
assert exercise_3_gap_solution(-1, 4, 10) == (4, 0)
assert exercise_3_gap_solution(0, 10, 10) == (9, 9)
assert exercise_3_gap_solution(0, 9, 10) == (4, 4)
assert exercise_3_gap_solution(4, 9, 10) == (2, 6)
for n in range(1, 12):
    for l in range(-1, n):
        for r in range(l + 1, n + 1):
            assert exercise_3_gap_solution(l, r, n) == gap_info(l, r, n), (l, r, n)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
class ExerciseRoom4Solution:
    def __init__(self, n):
        self.n = n
        self.nxt = {-1: n}
        self.prev = {n: -1}
        self.heap = []
        self._push(-1, n)

    def _push(self, l, r):
        heapq.heappush(self.heap, (-exercise_3_gap_solution(l, r, self.n)[0], l, r))

    def seat(self):
        while True:
            _, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:
                break
        pos = exercise_3_gap_solution(l, r, self.n)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos)
        self._push(pos, r)
        return pos

    def leave(self, p):
        l, r = self.prev.pop(p), self.nxt.pop(p)
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)

ExerciseRoom4Solution.__name__ = "ExamRoom"
assert run_ops(ExerciseRoom4Solution, example_ops, example_args) == example_expected
rng = random.Random(4)
for n in range(1, 15):
    for _ in range(20):
        ops, args = gen_ops(rng, n, rng.randint(1, 40))
        assert run_ops(ExerciseRoom4Solution, ops, args) == run_ops(brute_cls, ops, args), (n, ops, args)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
class ExerciseRoom5Solution:
    def __init__(self, n):
        self.n = n
        self.occ = []

    def seat(self):
        if not self.occ:
            self.occ.append(0)
            return 0
        best_d, best_pos = self.occ[0], 0
        for a, b in zip(self.occ, self.occ[1:]):
            d = (b - a) // 2
            if d >= best_d and d > 0:
                best_d, best_pos = d, (a + b + 1) // 2
        d = self.n - 1 - self.occ[-1]
        if d >= best_d and d > 0:
            best_d, best_pos = d, self.n - 1
        insort(self.occ, best_pos)
        return best_pos

    def leave(self, p):
        del self.occ[bisect_left(self.occ, p)]

ExerciseRoom5Solution.__name__ = "ExamRoomHigh"
rng = random.Random(5)
for n in range(2, 12):
    for _ in range(40):
        ops, args = gen_ops(rng, n, rng.randint(1, 40), ref_cls=ExamRoomHigh)
        ops = ["ExamRoomHigh"] + ops[1:]
        assert run_ops(ExerciseRoom5Solution, ops, args) == run_ops(high_brute, ops, args), (n, ops, args)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (đống các khoảng trống + xóa lười)

```python
import heapq

class ExamRoom:

    def __init__(self, n: int):
        self.n = n
        self.nxt = {-1: n}                 # láng giềng phải của mỗi người (và của gác -1)
        self.prev = {n: -1}                # láng giềng trái (và của gác n)
        self.heap = []                     # (-khoảng_cách, l, r)
        self._push(-1, n)

    def _best(self, l, r):                 # (khoảng cách, ghế) tốt nhất trong khoảng trống (l, r)
        n = self.n
        if l == -1:
            return (n, 0) if r == n else (r, 0)
        if r == n:
            return (n - 1 - l, n - 1)
        return ((r - l) // 2, (l + r) // 2)

    def _push(self, l, r):
        heapq.heappush(self.heap, (-self._best(l, r)[0], l, r))

    def seat(self) -> int:
        while True:
            _, l, r = heapq.heappop(self.heap)
            if self.nxt.get(l) == r:       # còn hợp lệ?
                break
        pos = self._best(l, r)[1]
        self.nxt[l], self.prev[pos] = pos, l
        self.nxt[pos], self.prev[r] = r, pos
        self._push(l, pos); self._push(pos, r)
        return pos

    def leave(self, p: int) -> None:
        l, r = self.prev.pop(p), self.nxt.pop(p)
        self.nxt[l], self.prev[r] = r, l
        self._push(l, r)
```

Công thức nhớ nhanh: khoảng trống `(l, r)` cho `(r, 0)` ở đầu hàng, `(n - 1 - l, n - 1)` ở cuối hàng, `((r - l) // 2, (l + r) // 2)` ở giữa; khóa so sánh `(-khoảng_cách, l)`.

### Checklist trước khi submit

- [ ] Biểu diễn bằng **khoảng trống** `(l, r)` với ký tự gác `-1` và `n` (không lưu từng ghế)?
- [ ] Ghế giữa là `(l + r) // 2` (ghế **nhỏ nhất** khi hòa), khoảng cách `(r - l) // 2`?
- [ ] Đầu/cuối hàng **không** chia đôi (`r` hoặc `n - 1 - l`)?
- [ ] Khóa đống `(-d, l, r)` (đống nhỏ nhất + phá hòa theo `l`)?
- [ ] Khi lấy từ đống: **kiểm tra** `nxt.get(l) == r` (xóa lười)?
- [ ] `leave(p)` gộp `(l, p)` và `(p, r)` thành `(l, r)` và nối lại `prev`/`nxt`?
- [ ] Trạng thái nằm trong `self` (không đặt ở cấp lớp)?
- [ ] Đã thử: `n = 1`, rời rồi ngồi lại ghế 0, ngồi đầy rồi rời hết, `n = 10^9`?
- [ ] Tên lớp/method đúng: `ExamRoom`, `seat()`, `leave(p)`; `seat` trả `int`, `leave` không trả gì?